In [1]:
import os
import glob
import numpy as np

In [ ]:
# Parámetros de configuración
current_dataset_dir = 'data'      # Carpeta del dataset original
desired_total = 620000            # Total deseado después de la ampliación
num_points = 5000                 # Cada espectro tiene 5000 puntos

# Rango de redshift para seleccionar nuevos espectros
z_min = 0.0
z_max = 8.0

# Cargar los redshifts existentes del dataset actual
current_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_450k_redshift_mmap.dat')
current_total = 460000  # Número actual de espectros
current_redshifts = np.memmap(current_redshift_path, dtype='float32', mode='r', shape=(current_total,))
existing_redshifts = set(current_redshifts.tolist())

# Cargar el nuevo dataset (3M espectros) desde archivos .dat
# Asumimos que los archivos nuevos se han guardado con np.memmap y tienen las siguientes rutas:
new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_mmap.dat')
new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_mmap.dat')
new_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_complete_redshift_mmap.dat')

new_total_available = 3372890  # Total de espectros en el nuevo dataset

# Cargar los memmaps del nuevo dataset en modo lectura
new_flux = np.memmap(new_flux_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_wavelength = np.memmap(new_wavelength_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_redshifts = np.memmap(new_redshift_path, dtype='float32', mode='r', shape=(new_total_available,))

new_flux_list = []
new_wavelength_list = []
new_redshift_list = []

print("Filtrando nuevos espectros del dataset de 3M...")
# Recorrer el nuevo dataset
for i in range(new_total_available):
    r = new_redshifts[i]
    # Filtrar por rango de redshift
    if r < z_min or r > z_max:
        continue
    # Evitar duplicados: se comprueba que el redshift no exista ya en el dataset original
    if r in existing_redshifts:
        continue
    # Si cumple ambas condiciones, se añade la información
    new_flux_list.append(new_flux[i, :].copy())         # .copy() para obtener un array independiente
    new_wavelength_list.append(new_wavelength[i, :].copy())
    new_redshift_list.append(r)
    existing_redshifts.add(r)  # Agregar para evitar duplicados posteriores
    
    # Si se alcanza el total deseado, se finaliza el filtrado
    if current_total + len(new_redshift_list) >= desired_total:
        break

print(f"Se han encontrado {len(new_redshift_list)} nuevos espectros en el rango de redshift [{z_min}, {z_max}].")

# Crear el nuevo dataset ampliado
new_total = current_total + len(new_redshift_list)
print(f"Dataset ampliado: {new_total} espectros.")

# Rutas para los nuevos archivos memmap actualizados
new_flux_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap.dat')
new_wavelength_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_wavelength_mmap.dat')
new_redshift_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_redshift_mmap.dat')

# Preasignar los memmaps para el dataset ampliado
flux_mmap_new = np.memmap(new_flux_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
wavelength_mmap_new = np.memmap(new_wavelength_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
redshift_mmap_new = np.memmap(new_redshift_mmap_path, dtype='float32', mode='w+', shape=(new_total,))

# Copiar los datos antiguos del dataset actual
current_flux_path = os.path.join(current_dataset_dir, 'spectra_data_450k_flux_mmap.dat')
current_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_450k_wavelength_mmap.dat')

flux_mmap_old = np.memmap(current_flux_path, dtype='float32', mode='r', shape=(current_total, num_points))
wavelength_mmap_old = np.memmap(current_wavelength_path, dtype='float32', mode='r', shape=(current_total, num_points))

# Copiar los datos del dataset original en los nuevos memmaps
flux_mmap_new[:current_total, :] = flux_mmap_old[:]
wavelength_mmap_new[:current_total, :] = wavelength_mmap_old[:]
redshift_mmap_new[:current_total] = current_redshifts[:]

# Añadir los nuevos espectros filtrados
for i, (flux_array, wave_array, r) in enumerate(zip(new_flux_list, new_wavelength_list, new_redshift_list)):
    idx = current_total + i
    flux_mmap_new[idx, :] = flux_array
    wavelength_mmap_new[idx, :] = wave_array
    redshift_mmap_new[idx] = r

# Asegurarse de que los cambios se guarden en disco
flux_mmap_new.flush()
wavelength_mmap_new.flush()
redshift_mmap_new.flush()

print("Se ha actualizado el dataset ampliado y se han guardado los nuevos archivos memmap.")

Filtrando nuevos espectros del dataset de 3M...
Se han encontrado 160000 nuevos espectros en el rango de redshift [0.0, 8.0].
Dataset ampliado: 620000 espectros.


In [ ]:
import os
import numpy as np

current_total = 620000
num_points = 5000
current_dataset_dir = 'data'
wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap.dat')
wavelength_memmap = np.memmap(wavelength_path, dtype='float32', mode='r')
print("Shape raw del memmap:", wavelength_memmap.shape)
print("Tamaño del archivo (bytes):", flux_mmap_new.shape)